In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

csv_path = Path("data/lots_of_repos.csv")

df = pd.read_csv(csv_path)

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nData types and non-null counts:")
df.info()

print("Total rows:", len(df))

if "url" in df.columns:
    print("Unique URLs:", df["url"].nunique())
    print("Duplicate URLs:", df["url"].duplicated().sum())

    duplicates = df[df["url"].duplicated(keep=False)]
    display(duplicates.head(10))

In [ ]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
})

missing.sort_values("missing_percent", ascending=False)


In [ ]:
numeric_cols = ["stars", "forks", "size"]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print(df[numeric_cols].dtypes)
print(df[numeric_cols].describe().round(2))

star_data = df["stars"].dropna()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.histplot(star_data, bins=40, ax=axes[0])
axes[0].set_title("Distribution of GitHub Stars")
axes[0].set_xlabel("Stars")

sns.histplot(np.log1p(star_data), bins=40, ax=axes[1])
axes[1].set_title("Distribution of log(1 + Stars)")
axes[1].set_xlabel("log(1 + stars)")

plt.tight_layout()
plt.show()

star_bins = pd.cut(
    df["stars"],
    bins=[-1, 99, 999, np.inf],
    labels=["Under 100", "100–999", "1,000+"]
)

print(star_bins.value_counts(sort=False))

star_bins.value_counts(sort=False).plot(
    kind="bar",
    color=["steelblue", "darkorange", "seagreen"]
)

plt.title("Repositories by Star-Count Range")
plt.xlabel("Star-count range")
plt.ylabel("Number of repositories")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
model_df = df.dropna(subset=["stars"]).copy()

star_threshold = model_df["stars"].median()

model_df["popularity"] = np.where(
    model_df["stars"] >= star_threshold,
    "high",
    "low"
)

print("Median star threshold:", star_threshold)
print("\nPopularity class counts:")
print(model_df["popularity"].value_counts())

print("\nPopularity class proportions:")
print(
    model_df["popularity"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .astype(str) + "%"
)

sns.countplot(
    data=model_df,
    x="popularity",
    order=["low", "high"]
)

plt.title("Low vs. High Popularity Repositories")
plt.xlabel("Popularity class")
plt.ylabel("Number of repositories")
plt.tight_layout()
plt.show()

In [ ]:
plot_df = model_df.dropna(subset=["stars", "forks"]).copy()

sns.scatterplot(
    data=plot_df,
    x="stars",
    y="forks",
    hue="popularity",
    alpha=0.65
)

plt.xscale("symlog", linthresh=1)
plt.yscale("symlog", linthresh=1)
plt.title("Stars vs. Forks by Popularity Class")
plt.xlabel("Stars")
plt.ylabel("Forks")
plt.tight_layout()
plt.show()

print(
    "Spearman correlation between stars and forks:",
    plot_df[["stars", "forks"]].corr(method="spearman").iloc[0, 1]
)

In [ ]:
sns.boxplot(
    data=model_df.dropna(subset=["size"]),
    x="popularity",
    y="size",
    order=["low", "high"]
)

plt.yscale("symlog", linthresh=1)
plt.title("Repository Size by Popularity Class")
plt.xlabel("Popularity class")
plt.ylabel("Repository size (log-like scale)")
plt.tight_layout()
plt.show()

In [ ]:

language_df = model_df.copy()
language_df["language"] = language_df["language"].fillna("Unknown")

top_languages = (
    language_df["language"]
    .value_counts()
    .head(12)
    .index
)

language_df = language_df[
    language_df["language"].isin(top_languages)
]

sns.countplot(
    data=language_df,
    y="language",
    hue="popularity",
    order=top_languages
)

plt.title("Popularity Classes Across the Most Common Languages")
plt.xlabel("Number of repositories")
plt.ylabel("Primary language")
plt.tight_layout()
plt.show()

In [ ]:
df["updated_at"] = pd.to_datetime(
    df["updated_at"],
    errors="coerce",
    utc=True
)

analysis_time = pd.Timestamp.now(tz="UTC")

df["days_since_update"] = (
    analysis_time - df["updated_at"]
).dt.total_seconds() / 86400

print("Missing or invalid dates:", df["updated_at"].isna().sum())
print("\nDays since update:")
print(df["days_since_update"].describe().round(2))

print(
    "\nRepositories with future update dates:",
    (df["days_since_update"] < 0).sum()
)

In [ ]:
model_df = df.dropna(
    subset=[
        "stars",
        "forks",
        "size",
        "days_since_update"
    ]
).copy()

nonnegative_cols = [
    "stars",
    "forks",
    "size",
    "days_since_update"
]

model_df = model_df[
    (model_df[nonnegative_cols] >= 0).all(axis=1)
].copy()

# Create the popularity target using the median from initial analysis
star_threshold = model_df["stars"].median()

model_df["popularity"] = np.where(
    model_df["stars"] >= star_threshold,
    "high",
    "low"
)

# Fill missing language values with an explicit category
model_df["language"] = (
    model_df["language"]
    .fillna("Unknown")
    .astype(str)
)

print("Rows available for modeling:", len(model_df))
print("Star threshold:", star_threshold)
print("\nTarget distribution:")
print(model_df["popularity"].value_counts())

print("\nPredictor preview:")
display(
    model_df[
        [
            "forks",
            "size",
            "days_since_update",
            "language",
            "stars",
            "popularity"
        ]
    ].head()
)

In [ ]:
numeric_features = [
    "forks",
    "size",
    "days_since_update"
]

X_numeric = model_df[numeric_features].copy()

X_numeric = np.log1p(X_numeric)

display(X_numeric.describe().round(3))

In [ ]:
X_language = pd.get_dummies(
    model_df["language"],
    prefix="language",
    dtype=float
)

print("Number of language categories:", model_df["language"].nunique())
print("Number of encoded columns:", X_language.shape[1])

display(X_language.head())

In [ ]:
X = pd.concat(
    [
        X_numeric.reset_index(drop=True),
        X_language.reset_index(drop=True)
    ],
    axis=1
)

y = model_df["popularity"].reset_index(drop=True)

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget counts:")
print(y.value_counts())

print("\nAny missing feature values?", X.isna().any().any())
print("Any infinite feature values?", np.isinf(X.to_numpy()).any())

display(X.head())

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("\nTraining class proportions:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTesting class proportions:")
print(y_test.value_counts(normalize=True).round(3))

In [ ]:
from sklearn.naive_bayes import GaussianNB

# Initialize and train the classifier
gnb = GaussianNB()

gnb.fit(X_train, y_train)

# Predict popularity on unseen test data
y_pred = gnb.predict(X_test)

print("Model trained successfully.")
print("Number of test predictions:", len(y_pred))
print("\nFirst 10 predictions:")
print(y_pred[:10])

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    confusion_matrix
)
import matplotlib.pyplot as plt

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {accuracy:.3f} ({accuracy:.1%})")

# Show precision, recall, F1-score, and class support
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

# Display the confusion matrix
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=["low", "high"]
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["low", "high"]
)

disp.plot(cmap="Blues", values_format="d")
plt.title("Gaussian Naive Bayes: Repository Popularity")
plt.show()

summary = (
    model_df
    .groupby("popularity")[["stars", "forks", "size", "days_since_update"]]
    .agg(["median", "mean"])
    .round(2)
)

display(summary)